# 02 · Train — embedding engine → GLORYS fine-tune (seed ensemble) + the models we compare against
Runtime: **GPU** (T4 works; L4/A100 faster). Every stage checkpoints to Drive each epoch and resumes after a
disconnect: just re-run the same cell.

| Cell | What | Needed for the leaderboard |
|---|---|---|
| Stage 0 | self-supervised embedding engine (no labels) | ✅ |
| Stage 2 | OceanEmbed fine-tune on GLORYS, one run per seed | ✅ (≥1 seed; 3 seeds = ensemble) |
| Published method | Attention 3D U-Net++ (Wang et al., ESSD 2026) retrained on **our** data | ✅ |
| Ridge | classical statistical baseline | ✅ |
| Ablation | input-window length | optional |

In [ ]:
# ── Setup: Drive (data) + GitHub (code) + deps ──────────────────────────────
from google.colab import drive, userdata
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/OceanEmbed'          # data, checkpoints, outputs live here
REPO = '/content/Sylithe_Ocean_Embedding'
import os, sys; os.makedirs(ROOT, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/karan27-dev/Sylithe_Ocean_Embedding {REPO}
sys.path.insert(0, REPO)
for m in [m for m in sys.modules if m.startswith('oceanembed')]:
    del sys.modules[m]                              # re-running this cell picks up freshly pulled code
!git -C {REPO} log --oneline -1
# Install only what this Colab image lacks (images differ between sessions), one package at a time, so pip
# never searches a large version space (that is what hung before).
import importlib, importlib.util, subprocess
NEED = {'xarray': 'xarray', 'zarr': 'zarr', 'dask': 'dask', 'netCDF4': 'netCDF4', 'scipy': 'scipy',
        'ee': 'earthengine-api', 'copernicusmarine': 'copernicusmarine', 'earthaccess': 'earthaccess', 'psutil': 'psutil'}
for mod, pkg in NEED.items():
    if importlib.util.find_spec(mod) is None:
        print('installing', pkg)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', pkg], check=True)
importlib.invalidate_caches()
for mod in NEED:
    print(f'{mod:17s}', getattr(importlib.import_module(mod), '__version__', 'ok'))

In [ ]:
import torch, shutil, numpy as np, pandas as pd
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: Runtime → Change runtime type')
from oceanembed import config as C, dataset as D, train as TR, baselines as B
# Drive is slow at tens of thousands of small files, fast at one big file. First session: copy the stores
# (~40 min, into a .part folder so an interrupted copy is never mistaken for a finished one), then save one
# archive to Drive. Every later session just unpacks that archive (a few minutes).
STORES, TAR = ['inputs.zarr', 'target.zarr'], f'{ROOT}/cache_inputs_target.tar'
if not all(os.path.exists(f'/content/{s}') for s in STORES):
    if os.path.exists(TAR):
        print('unpacking cached archive')
        !tar -xf {TAR} -C /content
    else:
        for s in STORES:
            if not os.path.exists(f'/content/{s}'):
                print('copying', s); shutil.rmtree(f'/content/{s}.part', ignore_errors=True)
                shutil.copytree(f'{ROOT}/{s}', f'/content/{s}.part'); os.rename(f'/content/{s}.part', f'/content/{s}')
        print('saving archive for fast future sessions')
        !tar -cf {TAR}.part -C /content inputs.zarr target.zarr && mv {TAR}.part {TAR}
STATS = f'{ROOT}/stats_v2.npz'                    # v2 = includes the input climatology for anomaly channels
if not os.path.exists(STATS):
    D.compute_stats('/content/inputs.zarr', '/content/target.zarr', STATS)
S = D.load_stats(STATS)
cfg = C.TrainConfig()                             # window 15, base 32, β-NLL 0.5, EMA 0.999
CK = f'{ROOT}/checkpoints/oceanembed_w{cfg.window}'
windows = lambda period, train, c=cfg: D.SurfaceWindows('/content/inputs.zarr', '/content/target.zarr', S, period,
                                                        c.window, train=train, var_dropout=c.var_dropout if train else 0,
                                                        crop=(c.crop_h, c.crop_w) if train and c.crop_h else None)
print('train days', len(windows(C.TRAIN, False)), '| val days', len(windows(C.VAL, False)))

## Stage 0 · Self-supervised embedding engine (masked surface autoencoder, no labels)

In [ ]:
ssl = lambda period, train: D.SurfaceWindows('/content/inputs.zarr', None, S, period, cfg.window, train=train,
                                             var_dropout=cfg.var_dropout if train else 0, patch_mask=0.5, need_target=False)
mae, _ = TR.new_models(cfg)
TR.fit(mae, ssl(C.TRAIN, True), ssl(C.VAL, False), cfg, 'ssl', CK, cfg.epochs_ssl)

## Stage 2 · OceanEmbed on GLORYS, all layers trainable (the paper's best transfer strategy)
One run per seed. Start with one seed; if GPU time allows, add two more: the ensemble of 3 usually lowers RMSE.
The printed `val RMSE °C` is against GLORYS 2022 (validation year, used only to pick the best epoch).

In [ ]:
SEEDS = [42]                     # e.g. [42, 7, 1234] for a 3-model ensemble
for seed in SEEDS:
    c = C.TrainConfig(**{**cfg.__dict__, 'seed': seed})
    _, net = TR.new_models(c)
    TR.load_encoder_from_ssl(net, f'{CK}/ssl_best.pt')
    TR.fit(net, windows(C.TRAIN, True, c), windows(C.VAL, False, c), c, 'glorys', f'{CK}/seed{seed}', c.epochs_glorys,
           depth_std=S['depth_std'])

In [ ]:
# Validation RMSE per depth (°C, vs GLORYS 2022) for the best epoch of each seed
rows = {f'seed {s}': torch.load(f'{CK}/seed{s}/glorys_best.pt', weights_only=False)['val']['per_depth'] for s in SEEDS
        if os.path.exists(f'{CK}/seed{s}/glorys_best.pt')}
pd.DataFrame({k: {d: v[d]['rmse'] for d in v} for k, v in rows.items()}).round(3)

## Published method · Attention 3D U-Net++ (Wang et al., ESSD 2026), retrained on our exact data
Same inputs, years, grid and scoring. This is the "we beat the published state of the art" comparison.

In [ ]:
cfg3 = C.TrainConfig(arch='attn_unetpp3d', base=16, batch_size=2)
_, net3 = TR.new_models(cfg3)
TR.fit(net3, windows(C.TRAIN, True, cfg3), windows(C.VAL, False, cfg3), cfg3, 'glorys',
       f'{ROOT}/checkpoints/attn_unetpp3d_w{cfg3.window}', cfg3.epochs_glorys, depth_std=S['depth_std'])

## Classical baseline · ridge regression (CPU, a few minutes)

In [ ]:
os.makedirs(f'{ROOT}/baselines', exist_ok=True)
B.Ridge(S).fit(windows(C.TRAIN, False), n_days=400).save(f'{ROOT}/baselines/ridge.npz')

## Optional ablation · input window length (paper: gain saturates ≈ 26 d)

In [ ]:
for w in [7, 26]:
    c = C.TrainConfig(window=w, epochs_glorys=20)
    _, n = TR.new_models(c); TR.load_encoder_from_ssl(n, f'{CK}/ssl_best.pt')   # encoder is window-agnostic
    print(w, TR.fit(n, windows(C.TRAIN, True, c), windows(C.VAL, False, c), c, 'glorys',
                    f'{ROOT}/checkpoints/ablate_w{w}', c.epochs_glorys, depth_std=S['depth_std']))